In [ ]:
# Example 1: inspect the top-level package surface.
import dagster_shared

public_names = [n for n in dir(dagster_shared) if not n.startswith('_')]
print(f"Public top-level names ({len(public_names)}):")
for n in public_names[:30]:
    print(f"  {n}: {type(getattr(dagster_shared, n)).__name__}")

assert hasattr(dagster_shared, '__path__'), 'dagster_shared should be a package'


In [ ]:
# Example 2: import the serdes submodule (dagster_shared's serialization layer).
import importlib

serdes_candidates = [
    'dagster_shared.serdes',
    'dagster_shared.check',
    'dagster_shared.error',
    'dagster_shared.utils',
]
imported = {}
for name in serdes_candidates:
    mod = importlib.import_module(name)
    imported[name] = mod
    print(f"OK   {name}: {len([x for x in dir(mod) if not x.startswith('_')])} public names")

# Every one of dagster_shared's core submodules must import (a missing one fails the cell)
assert list(imported) == serdes_candidates, sorted(set(serdes_candidates) - set(imported))


In [ ]:
# Example 3: exercise the `check` module (dagster's runtime type assertion helpers).
import dagster_shared.check as check

# Common dagster check helpers: str_param, int_param, bool_param, list_param, dict_param.
helpers = [n for n in dir(check) if n.endswith('_param') and not n.startswith('_')]
print(f"check.*_param helpers ({len(helpers)}):")
for h in helpers[:15]:
    print(f"  check.{h}")
assert {'str_param', 'int_param', 'bool_param', 'list_param', 'dict_param'} <= set(helpers), helpers

result = check.str_param('hello', 'greeting')
assert result == 'hello'
print("check.str_param('hello', 'greeting') -> 'hello'")

try:
    check.str_param(1, 'greeting')
except check.ParameterCheckError as e:
    print(f"check.str_param(1, 'greeting') rejected: {type(e).__name__}")
else:
    raise AssertionError("check.str_param accepted an int")

In [ ]:
# Example 4: confirm key runtime dependencies are importable alongside dagster_shared.
import importlib

runtime_deps = ['yaml', 'packaging', 'pydantic', 'typing_extensions', 'tomlkit', 'platformdirs']
missing = []
for dep in runtime_deps:
    try:
        mod = importlib.import_module(dep)
        ver = getattr(mod, '__version__', 'n/a')
        print(f"OK   {dep:20s} {ver}")
    except ImportError as e:
        missing.append(dep)
        print(f"MISS {dep}: {e}")

assert not missing, f'missing runtime deps: {missing}'


In [ ]:
# Example 5: round-trip overall sanity assertion.
import importlib
import pkgutil

import dagster_shared

submodule_names = [m.name for m in pkgutil.iter_modules(dagster_shared.__path__, prefix='dagster_shared.')]
print(f"Top-level submodules ({len(submodule_names)}):")
for n in submodule_names[:20]:
    print(f"  {n}")

# Ensure the package isn't a stub.
assert len(submodule_names) > 0, 'dagster_shared should expose submodules'
print('\nPASS: dagster_shared imported and is non-empty')
